In [1]:
# =====================================================
# MÓDULO 3
# Detección de Síntomas Tempranos de Diabetes
# Dataset: Early Stage Diabetes Risk Prediction
# Algoritmos:
#   - Logistic Regression
#   - Support Vector Machine (SVM)
#
# Revisión backend 2026-07-25: este notebook estaba bien planteado
# (Accuracy 0.92, Recall 0.91, AUC 0.97 con SVM). Único cambio: ruta
# del CSV y exportar el modelo ganador con joblib para el backend.
# =====================================================

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

df = pd.read_csv("../data/diabetes_data_upload.csv")
df = df.drop_duplicates()

print("Primeras filas")
print(df.head())
print("\nInformación")
print(df.info())
print("\nValores nulos")
print(df.isnull().sum())

X = df.drop("class", axis=1)
y = df["class"].map({"Positive": 1, "Negative": 0})

num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# =====================================================
# MODELO 1 - LOGISTIC REGRESSION
# =====================================================

modelo_lr = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("modelo", LogisticRegression(max_iter=1000))
])
modelo_lr.fit(X_train, y_train)
pred_lr = modelo_lr.predict(X_test)
prob_lr = modelo_lr.predict_proba(X_test)[:, 1]

print("\n===================================")
print("LOGISTIC REGRESSION")
print("===================================")
print("Accuracy :", round(accuracy_score(y_test, pred_lr), 4))
print("Precision:", round(precision_score(y_test, pred_lr), 4))
print("Recall   :", round(recall_score(y_test, pred_lr), 4))
print("F1 Score :", round(f1_score(y_test, pred_lr), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, prob_lr), 4))
print("\nMatriz de Confusión")
print(confusion_matrix(y_test, pred_lr))
print("\nClassification Report")
print(classification_report(y_test, pred_lr))

# =====================================================
# MODELO 2 - SUPPORT VECTOR MACHINE
# =====================================================

modelo_svm = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("modelo", SVC(kernel="rbf", probability=True, random_state=42))
])
modelo_svm.fit(X_train, y_train)
pred_svm = modelo_svm.predict(X_test)
prob_svm = modelo_svm.predict_proba(X_test)[:, 1]

print("\n===================================")
print("SUPPORT VECTOR MACHINE")
print("===================================")
print("Accuracy :", round(accuracy_score(y_test, pred_svm), 4))
print("Precision:", round(precision_score(y_test, pred_svm), 4))
print("Recall   :", round(recall_score(y_test, pred_svm), 4))
print("F1 Score :", round(f1_score(y_test, pred_svm), 4))
print("ROC AUC  :", round(roc_auc_score(y_test, prob_svm), 4))
print("\nMatriz de Confusión")
print(confusion_matrix(y_test, pred_svm))
print("\nClassification Report")
print(classification_report(y_test, pred_svm))

# =====================================================
# COMPARACIÓN FINAL Y SELECCIÓN DEL GANADOR
# =====================================================

resultados = pd.DataFrame({
    "Modelo": ["Logistic Regression", "Support Vector Machine"],
    "Accuracy": [accuracy_score(y_test, pred_lr), accuracy_score(y_test, pred_svm)],
    "Precision": [precision_score(y_test, pred_lr), precision_score(y_test, pred_svm)],
    "Recall": [recall_score(y_test, pred_lr), recall_score(y_test, pred_svm)],
    "F1": [f1_score(y_test, pred_lr), f1_score(y_test, pred_svm)],
    "ROC_AUC": [roc_auc_score(y_test, prob_lr), roc_auc_score(y_test, prob_svm)],
})

print("\n===================================")
print("COMPARACIÓN DE MODELOS")
print("===================================")
print(resultados)

candidatos = {"Logistic Regression": modelo_lr, "Support Vector Machine": modelo_svm}
ganador_nombre = resultados.sort_values("Recall", ascending=False).iloc[0]["Modelo"]
ganador_pipeline = candidatos[ganador_nombre]

print(f"\nModelo ganador por Recall: {ganador_nombre}")

joblib.dump(ganador_pipeline, "../models_artifacts/modelo3_sintomas_tempranos.joblib")
print("Guardado en ../models_artifacts/modelo3_sintomas_tempranos.joblib")
print("Columnas esperadas por el modelo:", list(X.columns))


Primeras filas
   Age Gender Polyuria Polydipsia sudden weight loss weakness Polyphagia  \
0   40   Male       No        Yes                 No      Yes         No   
1   58   Male       No         No                 No      Yes         No   
2   41   Male      Yes         No                 No      Yes        Yes   
3   45   Male       No         No                Yes      Yes        Yes   
4   60   Male      Yes        Yes                Yes      Yes        Yes   

  Genital thrush visual blurring Itching Irritability delayed healing  \
0             No              No     Yes           No             Yes   
1             No             Yes      No           No              No   
2             No              No     Yes           No             Yes   
3            Yes              No     Yes           No             Yes   
4             No             Yes     Yes          Yes             Yes   

  partial paresis muscle stiffness Alopecia Obesity     class  
0              No        

/tmp/ipykernel_114175/450511450.py:53: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns
/home/manix/Universidad/Inteligencia_artificial/Proyecto/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



SUPPORT VECTOR MACHINE
Accuracy : 0.9216
Precision: 0.9697
Recall   : 0.9143
F1 Score : 0.9412
ROC AUC  : 0.9661

Matriz de Confusión
[[15  1]
 [ 3 32]]

Classification Report
              precision    recall  f1-score   support

           0       0.83      0.94      0.88        16
           1       0.97      0.91      0.94        35

    accuracy                           0.92        51
   macro avg       0.90      0.93      0.91        51
weighted avg       0.93      0.92      0.92        51


COMPARACIÓN DE MODELOS
                   Modelo  Accuracy  Precision    Recall        F1   ROC_AUC
0     Logistic Regression  0.823529   0.933333  0.800000  0.861538  0.921429
1  Support Vector Machine  0.921569   0.969697  0.914286  0.941176  0.966071

Modelo ganador por Recall: Support Vector Machine
Guardado en ../models_artifacts/modelo3_sintomas_tempranos.joblib
Columnas esperadas por el modelo: ['Age', 'Gender', 'Polyuria', 'Polydipsia', 'sudden weight loss', 'weakness', 'Polyphagia'